# Chapter 1: Introduction

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 1, printed pp. 1-18, PDF pp. 19-36.

## Chapter Goal

This chapter is the runway for the rest of the course. Its purpose is not to give the formal definition of a topological manifold yet; that comes after the language of topological spaces is available. The goal here is to build a reliable mental model: a manifold is a space that may be globally curved, wrapped, glued, constrained, or high dimensional, while each sufficiently small neighborhood can still be described by ordinary Euclidean coordinates of one fixed dimension.

The computational lesson below translates that informal idea into inspectable objects. We will build a gallery of local coordinate models, compare a few spaces by invariants that survive homeomorphisms, and use a small classifier to see why topologists ask for lists of spaces together with invariants that distinguish the entries. The examples are deliberately modest. A circle, sphere, torus, cube boundary, graph of a function, and 3-sphere chart are enough to expose the main lesson: local coordinates are not the same thing as a global coordinate system, and visible shape is not the same thing as topological type.

Throughout this notebook, the word homeomorphic means topologically the same: there is a continuous bijection with continuous inverse. A topological invariant is a feature that a homeomorphism cannot change. Invariants are useful because they turn a negative statement, no homeomorphism exists, into something checkable: if two spaces have different invariant data, they cannot be homeomorphic.

In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-01-introduction/01-introduction.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-01-introduction/01-introduction.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-01-introduction/01-introduction.ipynb",
  "notebook_title": "Chapter 1: Introduction",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

| Book idea | Computational representation used here | What to inspect |
| --- | --- | --- |
| Local model on $\mathbb{R}^n$ | coordinate patches, parameter grids, stereographic projection samples | near one point, nearby parameters label nearby points without ambiguity |
| Dimension | number of independent parameters in the local coordinate record | a circle needs one local parameter; a surface needs two; the 3-sphere can be locally described by three |
| Homeomorphism | matched invariant signatures and deformation-friendly examples | sphere and cube boundary share the same coarse topological signature even though their geometry differs |
| Non-homeomorphism certificate | Euler characteristic and first Betti number comparison | a sphere-like surface and a torus-like surface disagree in hole data |
| Classification motivation | table of compact connected surface signatures | a useful classification pairs a standard list with computable distinguishing data |
| Application fields | examples routed to charts, meshes, constraints, and equations | manifolds appear as solution sets, configuration spaces, graphics patches, and spacetime models |

## Library Routing

The chapter is conceptual rather than computationally heavy, so the libraries are chosen to make topology visible without pretending that a single picture proves the theorem. Matplotlib is used for durable PNG diagrams: it is the right tool for a stable manifold gallery and invariant bar comparison. Plotly is used for the local chart atlas because rotating an embedded sphere while seeing its chart grid makes the local-global distinction easier to inspect than a static surface. NetworkX is used for the proof and classification dependency graph: the mathematical structure is a directed relationship among definitions, invariants, and classification claims. SymPy is used for exact formula checks such as $\chi=2-2g$ for orientable compact surfaces. Pandas is used only to display and save small tables of signatures and lab results.

## Visual Storyboard

1. **Manifold gallery:** compare curves, a circle, a sphere patch, and a torus. Inspection target: every example has local coordinates, but only some have a single global coordinate system.
2. **Local Euclidean chart atlas:** use stereographic projection on a sphere. Inspection target: a two-parameter plane grid becomes a curved surface patch, and the projection/inverse projection residual is numerically tiny away from the missing pole.
3. **Homeomorphism and invariant comparison:** compare sphere, cube boundary, torus, and double torus using Euler characteristic and first Betti number. Inspection target: geometric measurements can change, but these topological signatures do not.
4. **Proof/invariant scaffold:** make a dependency graph showing how local Euclidean charts lead to homeomorphism questions, and how invariants support non-homeomorphism and classification claims.
5. **Applied Lab:** classify sample compact connected surfaces from orientability and Euler characteristic. Inspection target: classification needs both a standard list and a checkable signature.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display


def locate_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'source_map.json').exists() and (candidate / 'utils').exists():
            return candidate
    raise RuntimeError('Could not locate Introduction-to-Topological-Manifolds book root')


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.source import unit_by_artifact_key  # noqa: E402
from utils.topology import cycle_rank_for_graph, euler_characteristic  # noqa: E402
from utils.validation import image_stats, relative  # noqa: E402

UNIT_KEY = 'chapter-01-introduction'
UNIT = unit_by_artifact_key(UNIT_KEY)
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / 'figures'
HTML = ARTIFACT_ROOT / 'html'
CHECKS = ARTIFACT_ROOT / 'checks'
TABLES = ARTIFACT_ROOT / 'tables'
SOURCE_SPAN = 'printed pp. 1-18, PDF pp. 19-36'


def display_path(path):
    return Path(os.path.relpath(Path(path), Path.cwd()))


def show(path, *, width=760, height=520):
    display_artifact(display_path(path), width=width, height=height)


assert UNIT['printed'] == '1-18'
assert UNIT['pdf'] == '19-36'
display(Markdown(f'Book root: `{BOOK_ROOT.name}`. Artifact root: `{relative(ARTIFACT_ROOT)}`. Source span: `{SOURCE_SPAN}`.'))

In [ ]:
storyboard = {
    'chapter_goal': 'Build an informal but testable model of topological manifolds as spaces that are locally Euclidean and studied up to homeomorphism.',
    'source_span_read': 'Chapter 1 Introduction, printed pp. 1-18, PDF pp. 19-36, inspected with pdftotext for terminology and structure only.',
    'library_routing': [
        {'concept': 'manifold gallery', 'representation': 'static chart-and-surface panels', 'library': 'matplotlib', 'why': 'durable labeled PNG for local-coordinate examples'},
        {'concept': 'local Euclidean chart', 'representation': 'interactive stereographic atlas', 'library': 'plotly', 'why': '3D rotation exposes a curved surface and a flat coordinate grid together'},
        {'concept': 'homeomorphism invariant comparison', 'representation': 'invariant bars and signature table', 'library': 'matplotlib, pandas, sympy', 'why': 'visible numeric signatures plus exact formula checks'},
        {'concept': 'proof and classification scaffold', 'representation': 'directed dependency graph', 'library': 'networkx', 'why': 'claims depend on definitions, invariants, and classification targets'},
    ],
    'visual_sequence': [
        {'order': 1, 'artifact': 'figures/manifold-gallery-local-models.png', 'inspection_target': 'local coordinate count for curves and surfaces', 'validation': 'nonblank PNG and table contains dimensions 1, 2, and 3'},
        {'order': 2, 'artifact': 'html/local-euclidean-chart-atlas.html', 'inspection_target': 'plane grid transferred to sphere patch', 'validation': 'stereographic projection round-trip residual below tolerance'},
        {'order': 3, 'artifact': 'figures/homeomorphism-invariant-comparison.png', 'inspection_target': 'sphere/cube agreement and torus disagreement', 'validation': 'Euler characteristic and Betti values match surface formulas'},
        {'order': 4, 'artifact': 'figures/proof-invariant-scaffold.png', 'inspection_target': 'how invariants support non-homeomorphism claims', 'validation': 'directed graph has the required dependency edges'},
        {'order': 5, 'artifact': 'tables/classification-lab.csv', 'inspection_target': 'classification from orientability plus Euler characteristic', 'validation': 'sample signatures classify to expected surface families'},
    ],
    'artifact_plan': {
        'figures': ['manifold-gallery-local-models.png', 'homeomorphism-invariant-comparison.png', 'proof-invariant-scaffold.png'],
        'html': ['local-euclidean-chart-atlas.html'],
        'checks': ['visual-storyboard.json', 'local-chart-checks.json', 'homeomorphism-invariant-checks.json', 'proof-scaffold-checks.json', 'classification-lab-checks.json', 'final-sanity.json'],
        'tables': ['manifold-gallery.csv', 'homeomorphism-invariant-comparison.csv', 'classification-lab.csv'],
    },
}

storyboard_path = save_json(storyboard, CHECKS / 'visual-storyboard.json')
display(pd.DataFrame(storyboard['visual_sequence']))
show(storyboard_path)

## Concept Section 1: Local Coordinates Before Formal Definitions

The informal definition in this chapter is intentionally provisional: think of an $n$-manifold as something that, near each of its points, can be described by $n$ real parameters. The phrase near each point is doing real work. A circle can be traced by an angle, but angle is not a one-to-one global coordinate because $0$ and $2\pi$ describe the same point. A sphere can be described by longitude and latitude away from the poles, but those coordinates break at the poles and wrap around along a meridian. The torus has a clean two-angle description as an embedded surface, yet each angle repeats.

The gallery below is not meant to settle all technical details. Instead, it separates three ideas that are easy to merge too early. First, a parameterization is a way to generate points. Second, a local chart is a reversible coordinate description on a small neighborhood. Third, a global model may require several charts even when the dimension is fixed everywhere. This distinction is the reason a later formal definition will avoid relying on one ambient Euclidean picture.

Inspect the highlighted neighborhoods. Each one has a small coordinate window that behaves like an open interval, a disk, or a ball in Euclidean space. The labels in the table record the local parameter count, not the dimension of the surrounding space in which we happened to draw the object.

In [ ]:
gallery_rows = [
    {'example': 'wavy embedded curve', 'ambient_space': 'R^2', 'local_model': 'open interval in R', 'dimension': 1, 'global_caution': 'one coordinate works only on a chosen arc'},
    {'example': 'circle', 'ambient_space': 'R^2', 'local_model': 'open interval in R', 'dimension': 1, 'global_caution': 'angle wraps around'},
    {'example': 'sphere surface', 'ambient_space': 'R^3', 'local_model': 'open disk in R^2', 'dimension': 2, 'global_caution': 'no single longitude-latitude chart covers the poles well'},
    {'example': 'torus surface', 'ambient_space': 'R^3', 'local_model': 'open disk in R^2', 'dimension': 2, 'global_caution': 'two angles are periodic'},
    {'example': '3-sphere near north pole', 'ambient_space': 'R^4', 'local_model': 'open ball in R^3', 'dimension': 3, 'global_caution': 'one coordinate must be solved from the constraint'},
]
gallery_table_path = save_csv(gallery_rows, TABLES / 'manifold-gallery.csv')

fig = plt.figure(figsize=(11, 8), constrained_layout=True)
ax_curve = fig.add_subplot(2, 2, 1)
x = np.linspace(-2.5, 2.5, 500)
y = 0.25 * np.sin(2.2 * x)
ax_curve.plot(x, y, color='#264653', lw=2)
mask = (x > -0.7) & (x < 0.7)
ax_curve.plot(x[mask], y[mask], color='#e76f51', lw=5, alpha=0.75)
ax_curve.scatter([0], [0], color='black', s=30, zorder=3)
ax_curve.annotate('one local parameter', xy=(0, 0), xytext=(-2.2, 0.55), arrowprops={'arrowstyle': '->'})
ax_curve.set_title('1D curve: local interval')
ax_curve.set_aspect('equal', adjustable='box')
ax_curve.axis('off')

ax_circle = fig.add_subplot(2, 2, 2)
theta = np.linspace(0, 2 * np.pi, 500)
ax_circle.plot(np.cos(theta), np.sin(theta), color='#2a9d8f', lw=2)
arc = np.linspace(0.2, 1.2, 90)
ax_circle.plot(np.cos(arc), np.sin(arc), color='#e76f51', lw=5, alpha=0.75)
ax_circle.scatter([np.cos(0.7)], [np.sin(0.7)], color='black', s=30)
ax_circle.text(-1.1, -1.28, 'angle is local; it repeats globally', fontsize=9)
ax_circle.set_title('Circle: local arc, global wrap')
ax_circle.set_aspect('equal')
ax_circle.axis('off')

ax_sphere = fig.add_subplot(2, 2, 3, projection='3d')
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0.12, np.pi - 0.12, 32)
uu, vv = np.meshgrid(u, v)
xs = np.cos(uu) * np.sin(vv)
ys = np.sin(uu) * np.sin(vv)
zs = np.cos(vv)
ax_sphere.plot_wireframe(xs, ys, zs, color='#457b9d', linewidth=0.45, alpha=0.55)
pu = np.linspace(-0.35, 0.35, 16)
pv = np.linspace(-0.35, 0.35, 16)
PU, PV = np.meshgrid(pu, pv)
patch_x = PU
patch_y = PV
patch_z = np.sqrt(np.maximum(0, 1 - PU**2 - PV**2))
ax_sphere.plot_surface(patch_x, patch_y, patch_z, color='#f4a261', alpha=0.75, linewidth=0)
ax_sphere.set_title('Sphere: two local parameters')
ax_sphere.set_axis_off()
ax_sphere.set_box_aspect((1, 1, 1))

ax_torus = fig.add_subplot(2, 2, 4, projection='3d')
a = np.linspace(0, 2 * np.pi, 64)
b = np.linspace(0, 2 * np.pi, 32)
aa, bb = np.meshgrid(a, b)
R, r = 1.6, 0.45
xt = (R + r * np.cos(bb)) * np.cos(aa)
yt = (R + r * np.cos(bb)) * np.sin(aa)
zt = r * np.sin(bb)
ax_torus.plot_surface(xt, yt, zt, color='#8ab17d', edgecolor='#3a5a40', linewidth=0.12, alpha=0.9)
ax_torus.set_title('Torus: two periodic parameters')
ax_torus.set_axis_off()
ax_torus.set_box_aspect((1, 1, 0.45))

gallery_png = save_matplotlib(fig, FIGURES / 'manifold-gallery-local-models.png')
plt.close(fig)

gallery_checks = {
    'examples': len(gallery_rows),
    'dimensions_present': sorted({row['dimension'] for row in gallery_rows}),
    'torus_major_radius_gt_minor_radius': R > r,
    'source_span': SOURCE_SPAN,
}
gallery_check_path = save_json(gallery_checks, CHECKS / 'manifold-gallery-checks.json')
display(pd.DataFrame(gallery_rows))
show(gallery_png, width=820)
show(gallery_table_path)
show(gallery_check_path)

## Concept Section 2: A Chart Is a Reversible Local Measurement

A chart should be read as a measurement device. It assigns Euclidean coordinates to points in a neighborhood, and it can reconstruct those points from the coordinates. For a sphere, stereographic projection gives a concrete example. Remove one pole, draw a line from that pole through a point on the sphere, and record where the line intersects a flat coordinate plane. The formula is not important here for its own sake. What matters is the reversible relationship on the part of the sphere where the chart is defined.

The Plotly artifact below places a flat grid below the sphere and sends that same grid to a curved patch on the sphere. Rotate the scene and notice the tension between local and global behavior. Locally, the grid behaves like ordinary $\mathbb{R}^2$ coordinates. Globally, one pole is missing from this chart, and large coordinate values run toward that missing point. This is why a manifold is not defined by one universal coordinate system. It is defined by the existence of enough compatible local coordinate systems to cover the space.

The check saved with the artifact verifies the round-trip identity numerically: chart coordinates are mapped to the sphere and then back to the chart plane. The residual is not a proof of the general theorem, but it is a useful computational invariant for this example.

In [ ]:
def inverse_stereographic(u, v):
    denom = 1 + u**2 + v**2
    return np.array([2 * u / denom, 2 * v / denom, (u**2 + v**2 - 1) / denom])


def stereographic_from_north(x, y, z):
    return np.array([x / (1 - z), y / (1 - z)])


phi = np.linspace(0, np.pi, 48)
theta = np.linspace(0, 2 * np.pi, 96)
TH, PH = np.meshgrid(theta, phi)
sphere_x = np.cos(TH) * np.sin(PH)
sphere_y = np.sin(TH) * np.sin(PH)
sphere_z = np.cos(PH)

fig_chart = go.Figure()
fig_chart.add_trace(go.Surface(x=sphere_x, y=sphere_y, z=sphere_z, colorscale='Blues', opacity=0.48, showscale=False, name='sphere'))

grid_values = np.linspace(-1.7, 1.7, 9)
line_values = np.linspace(-1.7, 1.7, 80)
for fixed in grid_values:
    sphere_line = inverse_stereographic(line_values, np.full_like(line_values, fixed))
    fig_chart.add_trace(go.Scatter3d(x=sphere_line[0], y=sphere_line[1], z=sphere_line[2], mode='lines', line={'color': '#e76f51', 'width': 4}, showlegend=False))
    sphere_line = inverse_stereographic(np.full_like(line_values, fixed), line_values)
    fig_chart.add_trace(go.Scatter3d(x=sphere_line[0], y=sphere_line[1], z=sphere_line[2], mode='lines', line={'color': '#2a9d8f', 'width': 4}, showlegend=False))
    fig_chart.add_trace(go.Scatter3d(x=line_values, y=np.full_like(line_values, fixed), z=np.full_like(line_values, -1.45), mode='lines', line={'color': '#e76f51', 'width': 2}, showlegend=False))
    fig_chart.add_trace(go.Scatter3d(x=np.full_like(line_values, fixed), y=line_values, z=np.full_like(line_values, -1.45), mode='lines', line={'color': '#2a9d8f', 'width': 2}, showlegend=False))

sample_uv = np.array([[-1.0, -0.7], [0.0, 0.0], [0.8, 0.5], [1.3, -1.1]])
for u0, v0 in sample_uv:
    p = inverse_stereographic(u0, v0)
    fig_chart.add_trace(go.Scatter3d(x=[u0, p[0]], y=[v0, p[1]], z=[-1.45, p[2]], mode='lines+markers', line={'color': '#1d3557', 'width': 3}, marker={'size': 3}, showlegend=False))

fig_chart.update_layout(
    title='Local Euclidean chart: a flat coordinate grid mapped onto a sphere',
    scene={
        'xaxis_title': 'x / chart u',
        'yaxis_title': 'y / chart v',
        'zaxis_title': 'z',
        'aspectmode': 'data',
        'camera': {'eye': {'x': 1.8, 'y': 1.7, 'z': 1.1}},
    },
    margin={'l': 0, 'r': 0, 't': 45, 'b': 0},
)

chart_html = save_plotly_html(fig_chart, HTML / 'local-euclidean-chart-atlas.html')
test_uv = np.array([(u0, v0) for u0 in np.linspace(-1.5, 1.5, 13) for v0 in np.linspace(-1.5, 1.5, 13)])
max_residual = 0.0
for u0, v0 in test_uv:
    point = inverse_stereographic(u0, v0)
    recovered = stereographic_from_north(*point)
    max_residual = max(max_residual, float(np.linalg.norm(recovered - np.array([u0, v0]))))

chart_checks = {
    'chart': 'stereographic projection from the north pole',
    'sample_count': int(len(test_uv)),
    'max_round_trip_residual': max_residual,
    'tolerance': 1e-12,
    'passes': max_residual < 1e-12,
}
chart_check_path = save_json(chart_checks, CHECKS / 'local-chart-checks.json')
show(chart_html, width=820, height=600)
show(chart_check_path)

## Concept Section 3: Homeomorphism Changes Geometry, Not Topology

The chapter stresses a practical asymmetry. To show that two spaces are homeomorphic, one can often write down or describe a specific reversible continuous deformation. A cube boundary and a round sphere have different angles, edge lengths, and curvature concentrations, but they are topologically the same surface. To show that two spaces are not homeomorphic, pictures are less reliable. One needs data that every homeomorphism preserves.

Euler characteristic is one such coarse invariant for compact surfaces. It does not see every possible distinction, but it immediately separates a sphere-like surface from a torus-like one. For orientable compact connected surfaces, the formula $\chi=2-2g$ links Euler characteristic to the number of handles $g$. The first Betti number $b_1=2g$ is a later algebraic-topology measurement of independent loop directions. In Chapter 1 this loop count is only motivational, but it captures the intuition behind rubber bands on a sphere versus rubber bands going around the two basic directions of a torus.

The comparison below deliberately includes both the sphere and cube boundary. They look geometrically different but have the same topological signature in this small table. The torus and double torus disagree, giving an invariant-based reason they cannot be homeomorphic to the sphere.

In [ ]:
surface_rows = [
    {'space': 'sphere boundary', 'cell_model': 'icosahedron boundary', 'V': 12, 'E': 30, 'F': 20, 'orientable': True, 'genus_or_crosscaps': 0, 'first_betti': 0, 'expected_class': 'sphere type'},
    {'space': 'cube boundary', 'cell_model': 'cube boundary', 'V': 8, 'E': 12, 'F': 6, 'orientable': True, 'genus_or_crosscaps': 0, 'first_betti': 0, 'expected_class': 'sphere type'},
    {'space': 'torus', 'cell_model': 'periodic 8 by 6 square grid', 'V': 48, 'E': 96, 'F': 48, 'orientable': True, 'genus_or_crosscaps': 1, 'first_betti': 2, 'expected_class': 'one-handle surface'},
    {'space': 'double torus', 'cell_model': 'one polygon with four edge pairs', 'V': 1, 'E': 4, 'F': 1, 'orientable': True, 'genus_or_crosscaps': 2, 'first_betti': 4, 'expected_class': 'two-handle surface'},
]
for row in surface_rows:
    row['euler_characteristic'] = euler_characteristic(row['V'], row['E'], row['F'])
    if row['orientable']:
        row['formula_chi'] = 2 - 2 * row['genus_or_crosscaps']
    else:
        row['formula_chi'] = 2 - row['genus_or_crosscaps']
    row['formula_matches_cells'] = row['euler_characteristic'] == row['formula_chi']

surface_df = pd.DataFrame(surface_rows)
surface_table_path = save_csv(surface_rows, TABLES / 'homeomorphism-invariant-comparison.csv')

g = sp.symbols('g', integer=True, nonnegative=True)
chi_formula = 2 - 2 * g
betti_formula = 2 * g
symbolic_checks = {
    'chi_orientable_genus_g': str(chi_formula),
    'b1_orientable_genus_g': str(betti_formula),
    'torus_chi_exact': int(chi_formula.subs(g, 1)),
    'double_torus_b1_exact': int(betti_formula.subs(g, 2)),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
colors = ['#457b9d', '#457b9d', '#e76f51', '#9d4edd']
axes[0].bar(surface_df['space'], surface_df['euler_characteristic'], color=colors)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('Euler characteristic')
axes[0].set_title('Homeomorphism invariant: Euler characteristic')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(surface_df['space'], surface_df['first_betti'], color=colors)
axes[1].set_ylabel('first Betti number, motivational loop count')
axes[1].set_title('Loop directions distinguish handles')
axes[1].tick_params(axis='x', rotation=20)
for ax in axes:
    ax.grid(axis='y', alpha=0.25)

invariant_png = save_matplotlib(fig, FIGURES / 'homeomorphism-invariant-comparison.png')
plt.close(fig)
invariant_checks = {
    'all_cell_counts_match_surface_formula': bool(surface_df['formula_matches_cells'].all()),
    'sphere_and_cube_share_signature': bool(surface_df.loc[0, ['euler_characteristic', 'first_betti']].to_dict() == surface_df.loc[1, ['euler_characteristic', 'first_betti']].to_dict()),
    'sphere_and_torus_differ': bool(surface_df.loc[0, 'euler_characteristic'] != surface_df.loc[2, 'euler_characteristic']),
    'symbolic_checks': symbolic_checks,
}
invariant_check_path = save_json(invariant_checks, CHECKS / 'homeomorphism-invariant-checks.json')
display(surface_df)
show(invariant_png, width=820)
show(surface_table_path)
show(invariant_check_path)

## Proof and Invariant Scaffold

A first course in manifolds quickly becomes a course in disciplined bookkeeping. The book begins with familiar pictures, but the later arguments need a way to decide which parts of a picture are topological and which parts are accidental geometry. The dependency graph below is a scaffold for that bookkeeping.

The starting node is local Euclidean behavior: a space has coordinate neighborhoods that look like ordinary Euclidean balls. Once spaces are compared by homeomorphism, statements about exact lengths, angles, and curvature are no longer protected. Statements about connectedness, compactness, loop classes, orientability, and Euler characteristic are designed to survive. The graph therefore routes the problem of classification through invariants. A positive classification theorem says every object in a chosen universe lands on a standard list. A separation theorem says two different entries on the list are not homeomorphic, usually because an invariant distinguishes them.

This graph is not a formal proof. It is a map of proof roles. In the chapters ahead, each node will be replaced by definitions and theorems: topological spaces, bases, compactness, connectedness, fundamental groups, covering spaces, cell complexes, and surface classification.

In [ ]:
G = nx.DiGraph()
edges = [
    ('local Euclidean charts', 'provisional manifold'),
    ('continuous maps', 'homeomorphism'),
    ('homeomorphism', 'topological property'),
    ('topological property', 'topological invariant'),
    ('topological invariant', 'non-homeomorphism certificate'),
    ('standard examples', 'classification list'),
    ('classification list', 'classification theorem'),
    ('topological invariant', 'classification theorem'),
    ('sphere/cube same signature', 'homeomorphism intuition'),
    ('sphere/torus different signature', 'non-homeomorphism certificate'),
]
G.add_edges_from(edges)
pos = {
    'local Euclidean charts': (0, 2),
    'provisional manifold': (2, 2),
    'continuous maps': (0, 1),
    'homeomorphism': (2, 1),
    'topological property': (4, 1),
    'topological invariant': (6, 1),
    'non-homeomorphism certificate': (8, 1.4),
    'standard examples': (2, -0.2),
    'classification list': (4, -0.2),
    'classification theorem': (7, -0.2),
    'sphere/cube same signature': (5.4, 2.1),
    'homeomorphism intuition': (7.6, 2.1),
    'sphere/torus different signature': (5.3, 0.0),
}
fig, ax = plt.subplots(figsize=(12, 5.8), constrained_layout=True)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=18, width=1.6, edge_color='#495057')
nx.draw_networkx_nodes(G, pos, ax=ax, node_color='#f1faee', edgecolors='#1d3557', linewidths=1.4, node_size=2400)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5)
ax.set_title('Proof scaffold: from local charts to classification by invariants')
ax.axis('off')
proof_png = save_matplotlib(fig, FIGURES / 'proof-invariant-scaffold.png')
plt.close(fig)

cycle_rank_example = cycle_rank_for_graph(vertex_count=1, edge_count=2, component_count=1)
proof_checks = {
    'node_count': G.number_of_nodes(),
    'edge_count': G.number_of_edges(),
    'has_invariant_to_certificate_edge': G.has_edge('topological invariant', 'non-homeomorphism certificate'),
    'has_classification_edge': G.has_edge('classification list', 'classification theorem'),
    'rose_graph_cycle_rank_two_loop_torus_spine': cycle_rank_example,
}
proof_check_path = save_json(proof_checks, CHECKS / 'proof-scaffold-checks.json')
show(proof_png, width=880)
show(proof_check_path)

## Applied Lab: A Tiny Surface Signature Classifier

The classification motivation in the chapter can be made concrete with a small lab. Suppose we restrict attention to compact connected surfaces without boundary, and suppose we already know two pieces of data: orientability and Euler characteristic. The full theorem is not proved here, but its shape can be practiced. If the surface is orientable, a nonnegative integer genus $g$ gives $\chi=2-2g$. If it is nonorientable, a positive integer number of crosscaps $k$ gives $\chi=2-k$.

The classifier below turns those formulas around. It is intentionally narrow: it does not decide whether an arbitrary input space really is a compact connected surface, and it does not compute orientability from a triangulation. Those are later mathematical tasks. Within the stated universe, however, it demonstrates the classification pattern: standard family plus invariant formula plus sanity checks.

Try changing the sample list after execution. For an orientable surface, odd Euler characteristic is rejected because $(2-\chi)/2$ would not be an integer genus. For a nonorientable surface, the formula permits projective plane type at $\chi=1$ and Klein bottle type at $\chi=0$. The lesson is that a classification theorem is useful only when its hypotheses, standard names, and invariants are all kept together.

In [ ]:
def classify_closed_connected_surface(orientable, chi):
    if orientable:
        numerator = 2 - chi
        if numerator < 0 or numerator % 2 != 0:
            return {'valid_signature': False, 'family': 'no orientable compact connected surface', 'parameter': None}
        genus = numerator // 2
        if genus == 0:
            name = 'sphere'
        elif genus == 1:
            name = 'torus'
        else:
            name = f'orientable genus {genus} surface'
        return {'valid_signature': True, 'family': name, 'parameter': int(genus)}
    crosscaps = 2 - chi
    if crosscaps < 1:
        return {'valid_signature': False, 'family': 'no nonorientable compact connected surface', 'parameter': None}
    if crosscaps == 1:
        name = 'projective plane'
    elif crosscaps == 2:
        name = 'Klein bottle'
    else:
        name = f'nonorientable {crosscaps}-crosscap surface'
    return {'valid_signature': True, 'family': name, 'parameter': int(crosscaps)}


lab_inputs = [
    {'sample': 'round sphere model', 'orientable': True, 'chi': 2},
    {'sample': 'one-handled surface', 'orientable': True, 'chi': 0},
    {'sample': 'two-handled surface', 'orientable': True, 'chi': -2},
    {'sample': 'projective-plane type', 'orientable': False, 'chi': 1},
    {'sample': 'Klein-bottle type', 'orientable': False, 'chi': 0},
    {'sample': 'invalid orientable odd chi', 'orientable': True, 'chi': 1},
]
lab_rows = []
for item in lab_inputs:
    classification = classify_closed_connected_surface(item['orientable'], item['chi'])
    lab_rows.append({**item, **classification})

lab_df = pd.DataFrame(lab_rows)
lab_table_path = save_csv(lab_rows, TABLES / 'classification-lab.csv')
lab_checks = {
    'row_count': len(lab_rows),
    'sphere_classified': lab_rows[0]['family'] == 'sphere',
    'torus_classified': lab_rows[1]['family'] == 'torus',
    'invalid_signature_rejected': lab_rows[-1]['valid_signature'] is False,
    'orientable_formula': 'chi = 2 - 2g',
    'nonorientable_formula': 'chi = 2 - k',
}
lab_check_path = save_json(lab_checks, CHECKS / 'classification-lab-checks.json')
display(lab_df)
show(lab_table_path)
show(lab_check_path)

## Final Sanity Checks

The final cell checks the notebook as a reproducible teaching object. It asserts that every planned artifact exists and is nonempty, that the PNGs are not blank, that the stereographic chart computation passes its residual tolerance, and that the invariant formulas still distinguish the intended examples. These checks do not replace the mathematics; they protect the computational translation from silent breakage.

In [ ]:
planned_artifacts = [
    storyboard_path,
    gallery_png,
    gallery_table_path,
    gallery_check_path,
    chart_html,
    chart_check_path,
    invariant_png,
    surface_table_path,
    invariant_check_path,
    proof_png,
    proof_check_path,
    lab_table_path,
    lab_check_path,
]
assert_artifacts(planned_artifacts, min_bytes=64)

for png in [gallery_png, invariant_png, proof_png]:
    stats = image_stats(png)
    assert stats['width'] >= 64 and stats['height'] >= 64
    assert stats['max_channel_stddev'] > 1.0

chart_loaded = json.loads(chart_check_path.read_text(encoding='utf-8'))
assert chart_loaded['passes']
assert chart_loaded['max_round_trip_residual'] < chart_loaded['tolerance']

invariant_loaded = json.loads(invariant_check_path.read_text(encoding='utf-8'))
assert invariant_loaded['all_cell_counts_match_surface_formula']
assert invariant_loaded['sphere_and_cube_share_signature']
assert invariant_loaded['sphere_and_torus_differ']

lab_loaded = json.loads(lab_check_path.read_text(encoding='utf-8'))
assert lab_loaded['sphere_classified'] and lab_loaded['torus_classified']
assert lab_loaded['invalid_signature_rejected']

final_sanity = {
    'artifact_count_checked_before_final_json': len(planned_artifacts),
    'pngs_checked_for_nonblank_content': [relative(path) for path in [gallery_png, invariant_png, proof_png]],
    'chart_round_trip_residual': chart_loaded['max_round_trip_residual'],
    'source_span': SOURCE_SPAN,
    'status': 'passed',
}
final_sanity_path = save_json(final_sanity, CHECKS / 'final-sanity.json')
assert_artifacts([final_sanity_path], min_bytes=64)
show(final_sanity_path)
display(Markdown(f"Final sanity checks passed for `{len(planned_artifacts) + 1}` artifacts."))

## Takeaways

A manifold is introduced here as a space with a fixed local Euclidean dimension. The local word is essential: circles, spheres, tori, constraint sets, configuration spaces, and higher-dimensional examples may require overlapping coordinate patches rather than one global coordinate system.

Homeomorphism is the chapter's provisional sameness relation. It ignores metric accidents such as exact length, angle, and roundedness, while preserving topological properties. That is why a cube boundary and a sphere can belong to the same topological type even though one has flat faces and sharp edges.

Invariants are the main tool for proving non-homeomorphism. Euler characteristic and loop-count intuition already separate sphere-like and torus-like surfaces. Later chapters turn that intuition into the fundamental group, covering spaces, cell complexes, homology, and rigorous surface classification.

The broader motivation is that manifolds arise whenever local coordinates describe a global object: algebraic solution sets, Riemann surfaces, graphics patches, rigid-body configurations, spacetime models, and compactified dimensions in physics. Chapter 1 is therefore not a detour; it names the recurring pattern the rest of the book will make precise.